# DeFi Protocol Risk Assessment Demo

This notebook demonstrates the comprehensive DeFi protocol risk assessment engine for the Algorand ecosystem.

## Features Demonstrated:
- **Cross-Protocol Exposure Analysis**: Analyze exposure across all major Algorand DeFi protocols
- **Concentration Risk Assessment**: Evaluate portfolio concentration and diversification
- **Systemic Risk Modeling**: Model cascade effects and contagion across protocols
- **Real-time Monitoring**: Generate alerts and recommendations for risk management

In [ ]:
# Import required libraries
import asyncio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

In [ ]:
# Import DeFi protocol risk assessment modules
import sys
sys.path.append('../')

from defi_protocol_risk.core.protocol_engine import DeFiProtocolRiskEngine
from defi_protocol_risk.core.protocol_exposure import ProtocolExposureAnalyzer
from defi_protocol_risk.core.concentration_risk import ConcentrationRiskAnalyzer
from defi_protocol_risk.core.systemic_risk import SystemicRiskAnalyzer

## 1. Initialize Risk Assessment Engine

In [ ]:
# Initialize the DeFi protocol risk engine
risk_engine = DeFiProtocolRiskEngine()

print("DeFi Protocol Risk Assessment Engine initialized successfully!")
print(f"Configuration loaded with {len(risk_engine.config.get('algorand_protocols', {}))} Algorand protocols")

## 2. Sample Portfolio Scenarios

Let's analyze different portfolio scenarios to demonstrate various risk profiles.

In [ ]:
# Define sample portfolios with different risk profiles

# Scenario 1: Well-diversified portfolio
balanced_portfolio = {
    'algofi': 30000,        # $30k in Algofi (established lending)
    'folks_finance': 25000,  # $25k in Folks Finance (lending)
    'tinyman': 20000,       # $20k in Tinyman (DEX)
    'pact': 15000,          # $15k in Pact (DEX)
    'humble_defi': 10000    # $10k in Humble DeFi (staking)
}

# Scenario 2: High concentration portfolio
concentrated_portfolio = {
    'algofi': 70000,        # $70k in Algofi (70% concentration)
    'tinyman': 20000,       # $20k in Tinyman
    'pact': 10000           # $10k in Pact
}

# Scenario 3: High-risk portfolio
risky_portfolio = {
    'yieldly': 50000,       # $50k in Yieldly (higher risk)
    'algomint': 30000,      # $30k in AlgoMint (bridge risk)
    'humble_defi': 20000    # $20k in Humble DeFi
}

portfolios = {
    'Balanced Portfolio': balanced_portfolio,
    'Concentrated Portfolio': concentrated_portfolio,
    'Risky Portfolio': risky_portfolio
}

print("Portfolio scenarios defined:")
for name, portfolio in portfolios.items():
    total_value = sum(portfolio.values())
    print(f"  {name}: ${total_value:,} across {len(portfolio)} protocols")

## 3. Comprehensive Risk Assessment

Let's perform comprehensive risk assessments for each portfolio scenario.

In [ ]:
# Perform risk assessments for all portfolios
assessments = {}

for portfolio_name, positions in portfolios.items():
    print(f"\nAnalyzing {portfolio_name}...")
    
    # Run comprehensive assessment
    assessment = await risk_engine.assess_portfolio_risk(positions)
    assessments[portfolio_name] = assessment
    
    print(f"  Overall Risk Score: {assessment.overall_portfolio_score:.2f}")
    print(f"  Risk Level: {assessment.risk_level}")
    print(f"  Confidence Score: {assessment.confidence_score:.2f}")

print("\nAll assessments completed successfully!")

## 4. Risk Metrics Visualization

Let's visualize the key risk metrics across all portfolios.

In [ ]:
# Extract key metrics for visualization
metrics_data = []

for portfolio_name, assessment in assessments.items():
    metrics = {
        'Portfolio': portfolio_name,
        'Overall Risk Score': assessment.overall_portfolio_score,
        'Concentration Risk': assessment.concentration_assessment.concentration_metrics.herfindahl_index,
        'Diversification Score': assessment.concentration_assessment.diversification_analysis.diversification_score,
        'Systemic Risk': assessment.systemic_assessment.systemic_risk_score,
        'Protocol Count': assessment.portfolio_metrics.protocol_count,
        'Category Count': assessment.portfolio_metrics.category_count,
        'Total Exposure': assessment.portfolio_metrics.total_exposure_usd
    }
    metrics_data.append(metrics)

# Create DataFrame for easier manipulation
metrics_df = pd.DataFrame(metrics_data)
print("Risk Metrics Summary:")
print(metrics_df.round(3))

In [ ]:
# Create comprehensive risk visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('DeFi Portfolio Risk Assessment Comparison', fontsize=16, fontweight='bold')

# 1. Overall Risk Scores
ax1 = axes[0, 0]
bars1 = ax1.bar(metrics_df['Portfolio'], metrics_df['Overall Risk Score'], 
                color=['green', 'orange', 'red'], alpha=0.7)
ax1.set_title('Overall Risk Score')
ax1.set_ylabel('Risk Score (0-1)')
ax1.set_ylim(0, 1)
for bar, score in zip(bars1, metrics_df['Overall Risk Score']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{score:.2f}', ha='center', va='bottom', fontweight='bold')

# 2. Concentration vs Diversification
ax2 = axes[0, 1]
x = np.arange(len(metrics_df))
width = 0.35
ax2.bar(x - width/2, metrics_df['Concentration Risk'], width, label='Concentration Risk', alpha=0.7)
ax2.bar(x + width/2, metrics_df['Diversification Score'], width, label='Diversification Score', alpha=0.7)
ax2.set_title('Concentration vs Diversification')
ax2.set_ylabel('Score (0-1)')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_df['Portfolio'])
ax2.legend()

# 3. Systemic Risk
ax3 = axes[0, 2]
bars3 = ax3.bar(metrics_df['Portfolio'], metrics_df['Systemic Risk'], 
                color=['lightblue', 'lightcoral', 'darkred'], alpha=0.7)
ax3.set_title('Systemic Risk Score')
ax3.set_ylabel('Risk Score (0-1)')
ax3.set_ylim(0, 1)
for bar, score in zip(bars3, metrics_df['Systemic Risk']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{score:.2f}', ha='center', va='bottom', fontweight='bold')

# 4. Protocol Distribution
ax4 = axes[1, 0]
ax4.bar(metrics_df['Portfolio'], metrics_df['Protocol Count'], 
        color=['skyblue', 'lightgreen', 'plum'], alpha=0.7)
ax4.set_title('Protocol Count')
ax4.set_ylabel('Number of Protocols')

# 5. Category Distribution
ax5 = axes[1, 1]
ax5.bar(metrics_df['Portfolio'], metrics_df['Category Count'], 
        color=['gold', 'silver', 'bronze'], alpha=0.7)
ax5.set_title('Category Count')
ax5.set_ylabel('Number of Categories')

# 6. Total Exposure
ax6 = axes[1, 2]
bars6 = ax6.bar(metrics_df['Portfolio'], metrics_df['Total Exposure'] / 1000, 
                color=['lightsteelblue', 'lightcyan', 'lightyellow'], alpha=0.7)
ax6.set_title('Total Exposure')
ax6.set_ylabel('Exposure ($K)')
for bar, exposure in zip(bars6, metrics_df['Total Exposure']):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'${exposure/1000:.0f}K', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Protocol Exposure Analysis

Let's analyze the detailed protocol exposure for each portfolio.

In [ ]:
# Analyze protocol exposure details
def analyze_protocol_exposure(assessment, portfolio_name):
    print(f"\n=== {portfolio_name} Protocol Exposure Analysis ===")
    
    exposure_analysis = assessment.exposure_analysis
    
    print(f"Total Exposure: ${exposure_analysis.total_exposure_usd:,.2f}")
    print(f"Concentration Risk: {exposure_analysis.concentration_risk_score:.2f}")
    print(f"Diversification Score: {exposure_analysis.diversification_score:.2f}")
    print(f"Systemic Risk: {exposure_analysis.systemic_risk_score:.2f}")
    print(f"Overall Risk Level: {exposure_analysis.risk_level}")
    
    print("\nProtocol Breakdown:")
    for exp in exposure_analysis.protocol_exposures:
        print(f"  {exp.protocol_name} ({exp.category}):")
        print(f"    Exposure: ${exp.user_exposure_usd:,.0f} ({exp.exposure_percentage:.1f}%)")
        print(f"    Risk Tier: {exp.risk_tier}")
        print(f"    Concentration Risk: {exp.concentration_risk_score:.2f}")
        print(f"    Liquidity Risk: {exp.liquidity_risk_score:.2f}")
    
    return exposure_analysis

# Analyze each portfolio
for portfolio_name, assessment in assessments.items():
    analyze_protocol_exposure(assessment, portfolio_name)

In [ ]:
# Create protocol exposure visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Protocol Exposure Distribution by Portfolio', fontsize=16, fontweight='bold')

for idx, (portfolio_name, assessment) in enumerate(assessments.items()):
    ax = axes[idx]
    
    # Extract protocol data
    protocols = []
    exposures = []
    colors = []
    
    for exp in assessment.exposure_analysis.protocol_exposures:
        protocols.append(exp.protocol_name)
        exposures.append(exp.exposure_percentage)
        
        # Color by risk tier
        if exp.risk_tier == 'tier_1':
            colors.append('green')
        elif exp.risk_tier == 'tier_2':
            colors.append('orange')
        else:
            colors.append('red')
    
    # Create pie chart
    wedges, texts, autotexts = ax.pie(exposures, labels=protocols, autopct='%1.1f%%', 
                                      colors=colors, startangle=90)
    ax.set_title(f'{portfolio_name}\n(Risk: {assessment.risk_level})')
    
    # Enhance text
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

## 6. Systemic Risk Analysis

Let's examine the systemic risk assessment and cascade scenarios.

In [ ]:
# Analyze systemic risk for balanced portfolio (most comprehensive)
balanced_assessment = assessments['Balanced Portfolio']
systemic_assessment = balanced_assessment.systemic_assessment

print("=== Systemic Risk Analysis ===")
print(f"Overall Systemic Risk Score: {systemic_assessment.systemic_risk_score:.2f}")
print(f"Network Stability Score: {systemic_assessment.network_stability_score:.2f}")
print(f"Interconnectedness Index: {systemic_assessment.interconnectedness_index:.2f}")
print(f"Ecosystem Health: {systemic_assessment.overall_ecosystem_health:.2f}")
print(f"Risk Level: {systemic_assessment.risk_level}")

print("\n=== Network Nodes (Protocols) ===")
for node in systemic_assessment.network_nodes:
    print(f"{node.protocol_name}:")
    print(f"  Systemic Importance: {node.systemic_importance:.2f}")
    print(f"  Interconnectedness: {node.interconnectedness:.2f}")
    print(f"  Failure Probability: {node.failure_probability:.2f}")
    print(f"  Cascade Vulnerability: {node.cascade_vulnerability:.2f}")

print(f"\n=== Cascade Scenarios ({len(systemic_assessment.cascade_scenarios)} total) ===")
for i, scenario in enumerate(systemic_assessment.cascade_scenarios[:3]):  # Show top 3
    print(f"Scenario {i+1}: {scenario.trigger_protocol}")
    print(f"  Affected Protocols: {len(scenario.affected_protocols)}")
    print(f"  Cascade Probability: {scenario.cascade_probability:.2f}")
    print(f"  TVL Impact: ${scenario.total_tvl_impact:,.0f}")
    print(f"  Economic Impact: ${scenario.economic_impact_usd:,.0f}")
    print(f"  Recovery Time: {scenario.recovery_time_days:.0f} days")

In [ ]:
# Visualize systemic risk network
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Systemic Risk Network Analysis', fontsize=16, fontweight='bold')

# 1. Protocol Systemic Importance
protocols = [node.protocol_name for node in systemic_assessment.network_nodes]
importance = [node.systemic_importance for node in systemic_assessment.network_nodes]
failure_prob = [node.failure_probability for node in systemic_assessment.network_nodes]

scatter = ax1.scatter(importance, failure_prob, 
                     s=[node.cascade_vulnerability * 1000 for node in systemic_assessment.network_nodes],
                     alpha=0.6, c=importance, cmap='RdYlGn_r')
ax1.set_xlabel('Systemic Importance')
ax1.set_ylabel('Failure Probability')
ax1.set_title('Protocol Risk-Importance Matrix\n(Size = Cascade Vulnerability)')

# Add protocol labels
for i, protocol in enumerate(protocols):
    ax1.annotate(protocol, (importance[i], failure_prob[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# 2. Cascade Impact Analysis
cascade_triggers = [scenario.trigger_protocol for scenario in systemic_assessment.cascade_scenarios[:5]]
cascade_probs = [scenario.cascade_probability for scenario in systemic_assessment.cascade_scenarios[:5]]
cascade_impacts = [scenario.total_tvl_impact / 1000000 for scenario in systemic_assessment.cascade_scenarios[:5]]  # In millions

bars = ax2.bar(range(len(cascade_triggers)), cascade_impacts, 
               color=plt.cm.Reds([prob for prob in cascade_probs]), alpha=0.7)
ax2.set_xlabel('Cascade Trigger Protocol')
ax2.set_ylabel('TVL Impact ($M)')
ax2.set_title('Top Cascade Scenarios\n(Color = Probability)')
ax2.set_xticks(range(len(cascade_triggers)))
ax2.set_xticklabels(cascade_triggers, rotation=45, ha='right')

# Add probability labels
for i, (bar, prob) in enumerate(zip(bars, cascade_probs)):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{prob:.0%}', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Risk Alerts and Recommendations

Let's examine the risk alerts and recommendations generated for each portfolio.

In [ ]:
# Analyze risk alerts and recommendations
def display_alerts_and_recommendations(assessment, portfolio_name):
    print(f"\n=== {portfolio_name} Risk Alerts and Recommendations ===")
    
    print(f"\n📊 Overall Portfolio Score: {assessment.overall_portfolio_score:.2f}")
    print(f"🚨 Risk Level: {assessment.risk_level}")
    print(f"🔍 Confidence Score: {assessment.confidence_score:.2f}")
    print(f"📅 Next Review Date: {assessment.next_review_date.strftime('%Y-%m-%d')}")
    
    # Risk Alerts
    if assessment.risk_alerts:
        print(f"\n🚨 Risk Alerts ({len(assessment.risk_alerts)} total):")
        for i, alert in enumerate(assessment.risk_alerts[:5], 1):  # Show top 5
            severity_emoji = {
                'HIGH': '🔴',
                'MEDIUM': '🟡',
                'LOW': '🟢'
            }.get(alert.severity, '⚪')
            print(f"  {i}. {severity_emoji} {alert.severity}: {alert.message}")
            print(f"     Action: {alert.recommended_action}")
    else:
        print("\n✅ No critical risk alerts detected.")
    
    # Recommendations
    if assessment.recommendations:
        print(f"\n💡 Recommendations ({len(assessment.recommendations)} total):")
        for i, rec in enumerate(assessment.recommendations, 1):
            priority = "🔥" if "URGENT" in rec else "⚡" if "HIGH" in rec else "📋"
            print(f"  {i}. {priority} {rec}")
    else:
        print("\n✅ Portfolio is well-optimized. Continue monitoring.")

# Display for all portfolios
for portfolio_name, assessment in assessments.items():
    display_alerts_and_recommendations(assessment, portfolio_name)

## 8. Risk Score Comparison and Trends

Let's create a comprehensive comparison of risk scores across portfolios.

In [ ]:
# Create comprehensive risk score comparison
risk_categories = [
    'Overall Portfolio Risk',
    'Concentration Risk',
    'Systemic Risk',
    'Liquidity Risk',
    'Diversification Score'
]

# Extract risk scores for comparison
portfolio_names = list(assessments.keys())
risk_scores = {
    'Overall Portfolio Risk': [1 - assess.overall_portfolio_score for assess in assessments.values()],
    'Concentration Risk': [assess.concentration_assessment.overall_risk_score for assess in assessments.values()],
    'Systemic Risk': [assess.systemic_assessment.systemic_risk_score for assess in assessments.values()],
    'Liquidity Risk': [assess.portfolio_metrics.liquidity_coverage_ratio for assess in assessments.values()],
    'Diversification Score': [assess.concentration_assessment.diversification_analysis.diversification_score for assess in assessments.values()]
}

# Create radar chart for risk comparison
from math import pi

fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(projection='polar'))

# Number of variables
N = len(risk_categories)

# Compute angle for each axis
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Colors for each portfolio
colors = ['green', 'orange', 'red']
alphas = [0.3, 0.3, 0.3]

# Plot each portfolio
for i, portfolio_name in enumerate(portfolio_names):
    values = [risk_scores[category][i] for category in risk_categories]
    values += values[:1]  # Complete the circle
    
    ax.plot(angles, values, 'o-', linewidth=2, label=portfolio_name, color=colors[i])
    ax.fill(angles, values, alpha=alphas[i], color=colors[i])

# Add category labels
ax.set_xticks(angles[:-1])
ax.set_xticklabels(risk_categories)
ax.set_ylim(0, 1)
ax.set_title('Portfolio Risk Profile Comparison\n(Larger area = Higher Risk)', 
             size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Portfolio Optimization Suggestions

Let's generate specific portfolio optimization suggestions based on the risk analysis.

In [ ]:
# Generate portfolio optimization suggestions
def generate_optimization_suggestions(assessment, current_positions, portfolio_name):
    print(f"\n=== {portfolio_name} Optimization Suggestions ===")
    
    total_value = sum(current_positions.values())
    concentration = assessment.concentration_assessment
    
    # Current state
    print(f"Current Portfolio Value: ${total_value:,}")
    print(f"Current Risk Level: {assessment.risk_level}")
    print(f"Current Risk Score: {assessment.overall_portfolio_score:.2f}")
    
    # Optimization suggestions
    suggestions = []
    
    # Check concentration
    max_exposure = max((value/total_value)*100 for value in current_positions.values())
    if max_exposure > 40:
        max_protocol = max(current_positions.items(), key=lambda x: x[1])
        target_amount = total_value * 0.35  # Target 35% max
        reduction = max_protocol[1] - target_amount
        suggestions.append({
            'type': 'Reduce Concentration',
            'action': f'Reduce {max_protocol[0]} by ${reduction:,.0f} (to 35% max)',
            'benefit': 'Lower concentration risk',
            'priority': 'HIGH'
        })
    
    # Check diversification
    current_protocols = len(current_positions)
    if current_protocols < 4:
        missing_protocols = 4 - current_protocols
        suggestions.append({
            'type': 'Increase Diversification',
            'action': f'Add {missing_protocols} more protocol(s) to portfolio',
            'benefit': 'Better risk distribution',
            'priority': 'MEDIUM'
        })
    
    # Check category diversification
    categories = set()
    for protocol in current_positions.keys():
        protocol_config = assessment.exposure_analysis.protocol_exposures
        for exp in protocol_config:
            if exp.protocol_name == protocol:
                categories.add(exp.category)
    
    if len(categories) < 3:
        suggestions.append({
            'type': 'Category Diversification',
            'action': 'Add exposure to different protocol categories (lending, DEX, staking)',
            'benefit': 'Reduce category correlation risk',
            'priority': 'MEDIUM'
        })
    
    # Check tier distribution
    tier_3_exposure = 0
    for exp in assessment.exposure_analysis.protocol_exposures:
        if exp.risk_tier == 'tier_3':
            tier_3_exposure += exp.exposure_percentage
    
    if tier_3_exposure > 25:
        suggestions.append({
            'type': 'Tier Optimization',
            'action': f'Reduce Tier 3 protocol exposure from {tier_3_exposure:.1f}% to <25%',
            'benefit': 'Lower protocol risk',
            'priority': 'HIGH'
        })
    
    # Display suggestions
    if suggestions:
        print("\n📈 Optimization Suggestions:")
        for i, suggestion in enumerate(suggestions, 1):
            priority_emoji = {'HIGH': '🔥', 'MEDIUM': '⚡', 'LOW': '📋'}[suggestion['priority']]
            print(f"  {i}. {priority_emoji} {suggestion['type']}")
            print(f"     Action: {suggestion['action']}")
            print(f"     Benefit: {suggestion['benefit']}")
            print(f"     Priority: {suggestion['priority']}")
    else:
        print("\n✅ Portfolio is well-optimized! No major changes needed.")
    
    return suggestions

# Generate suggestions for each portfolio
all_suggestions = {}
for portfolio_name, positions in portfolios.items():
    assessment = assessments[portfolio_name]
    suggestions = generate_optimization_suggestions(assessment, positions, portfolio_name)
    all_suggestions[portfolio_name] = suggestions

## 10. Real-time Monitoring Setup

Let's demonstrate how to set up real-time monitoring for portfolio risk changes.

In [ ]:
# Simulate real-time monitoring
async def simulate_monitoring_scenario(risk_engine, initial_positions):
    print("=== Real-time Monitoring Simulation ===")
    
    # Initial assessment
    initial_assessment = await risk_engine.assess_portfolio_risk(initial_positions)
    print(f"Initial Risk Score: {initial_assessment.overall_portfolio_score:.2f}")
    print(f"Initial Risk Level: {initial_assessment.risk_level}")
    
    # Simulate market changes
    market_scenarios = [
        {
            'name': 'Protocol TVL Drop',
            'description': 'Algofi TVL drops 20%',
            'impact': 'Increased concentration risk'
        },
        {
            'name': 'New Security Incident',
            'description': 'Security incident in bridge protocol',
            'impact': 'Increased systemic risk'
        },
        {
            'name': 'Governance Attack',
            'description': 'Governance token concentration attack',
            'impact': 'Increased governance risk'
        }
    ]
    
    print("\n📡 Monitoring Scenarios:")
    for i, scenario in enumerate(market_scenarios, 1):
        print(f"  {i}. {scenario['name']}")
        print(f"     Description: {scenario['description']}")
        print(f"     Expected Impact: {scenario['impact']}")
    
    # Monitoring recommendations
    print("\n🔍 Monitoring Recommendations:")
    monitoring_items = [
        "Track protocol TVL changes (>10% change triggers alert)",
        "Monitor governance token concentration shifts",
        "Watch for new security incidents or exploits",
        "Track cross-protocol correlation changes",
        "Monitor liquidity changes in key protocols",
        "Set up automated rebalancing triggers"
    ]
    
    for i, item in enumerate(monitoring_items, 1):
        print(f"  {i}. ⚡ {item}")
    
    return market_scenarios

# Run monitoring simulation
monitoring_scenarios = await simulate_monitoring_scenario(risk_engine, balanced_portfolio)

## 11. Export Assessment Reports

Let's export detailed assessment reports for each portfolio.

In [ ]:
# Export assessment reports
report_paths = {}

for portfolio_name, assessment in assessments.items():
    # Create filename
    filename = f"defi_risk_assessment_{portfolio_name.lower().replace(' ', '_')}.json"
    
    # Export report
    report_path = await risk_engine.export_assessment_report(assessment, filename)
    report_paths[portfolio_name] = report_path
    
    print(f"📄 {portfolio_name} report exported to: {report_path}")

print(f"\n✅ All {len(report_paths)} assessment reports exported successfully!")

## 12. Summary and Key Insights

Let's summarize the key insights from our DeFi protocol risk analysis.

In [ ]:
# Generate summary insights
print("=== DeFi Protocol Risk Assessment Summary ===")
print()

# Portfolio rankings
portfolio_rankings = sorted(
    [(name, assess.overall_portfolio_score, assess.risk_level) 
     for name, assess in assessments.items()],
    key=lambda x: x[1], reverse=True
)

print("🏆 Portfolio Risk Rankings (Best to Worst):")
for i, (name, score, level) in enumerate(portfolio_rankings, 1):
    emoji = "🥇" if i == 1 else "🥈" if i == 2 else "🥉"
    print(f"  {emoji} {i}. {name}: {score:.2f} ({level})")

print("\n📊 Key Risk Factors Identified:")
risk_factors = [
    "High concentration in single protocols (>40% exposure)",
    "Limited category diversification across DeFi sectors",
    "Systemic risk from protocol interconnectedness",
    "Exposure to higher-risk tier 3 protocols",
    "Governance token concentration vulnerabilities"
]

for i, factor in enumerate(risk_factors, 1):
    print(f"  {i}. ⚠️  {factor}")

print("\n💡 Best Practices Recommendations:")
best_practices = [
    "Maintain maximum 35% exposure to any single protocol",
    "Diversify across at least 3 protocol categories (lending, DEX, staking)",
    "Prioritize tier 1 protocols for major allocations (>60% of portfolio)",
    "Monitor protocol governance token concentration risks",
    "Set up automated monitoring for TVL and correlation changes",
    "Regular risk assessment reviews (weekly for high-risk portfolios)"
]

for i, practice in enumerate(best_practices, 1):
    print(f"  {i}. ✅ {practice}")

print("\n🎯 Risk Management Framework:")
framework_elements = [
    "Continuous Monitoring: Real-time tracking of protocol health",
    "Dynamic Rebalancing: Automated triggers for portfolio adjustments",
    "Stress Testing: Regular scenario analysis for crisis preparedness",
    "Early Warning System: Alerts for emerging risks and threats",
    "Compliance Integration: Regulatory risk monitoring and reporting"
]

for i, element in enumerate(framework_elements, 1):
    print(f"  {i}. 🛡️  {element}")

print("\n" + "="*60)
print("DeFi Protocol Risk Assessment Demo Completed Successfully!")
print("="*60)